# 🎬 Нейро-анимация кадров бесплатно: LTX-Video на Kaggle GPU

**Что делает ноутбук:** берёт готовые кадры (например, из папки `mpt-materials/` этого репозитория)
и **оживляет их** — настоящая генеративная анимация с движением внутри кадра, а не наезд камеры.

**Что нужно:**
1. Аккаунт Kaggle + **подтверждённый телефон** (без него нельзя включить интернет на GPU).
2. Справа в панели ноутбука: **Settings → Accelerator → GPU T4 x2** (или P100).
3. **Settings → Internet → On**.

**Что будет:** 4 клипа по ~4 секунды, вертикаль 704×1216 (9:16), затем склейка в один ролик
со русской озвучкой. Всё в пределах бесплатных ~30 GPU-часов в неделю.

**Ограничения, честно:** модель 2B distilled — быстрая и влезает в 16 ГБ, но картинка мягче,
чем у платных Veo/Kling. 13B даёт лучше, однако на T4 будет считать в разы дольше (можно
включить `offload_to_cpu`, см. последнюю ячейку).

## 1. Проверка окружения

Запустите ячейку: должны увидеть GPU и достаточно места (веса ~10 ГБ).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import shutil, torch
free = shutil.disk_usage("/kaggle/working").free / 1e9 if __import__("os").path.isdir("/kaggle/working") else 0
print(f"torch {torch.__version__}, cuda {torch.cuda.is_available()}")
print(f"свободно на диске: {free:.1f} ГБ")
assert torch.cuda.is_available(), "Включите GPU в Settings -> Accelerator"

## 2. Установка LTX-Video

Официальный репозиторий Lightricks (Apache-2.0). Ставим в режиме editable с extras `[inference]`.
Занимает ~2–3 минуты.

In [ ]:
!git clone -q https://github.com/Lightricks/LTX-Video.git /kaggle/working/LTX-Video
%cd /kaggle/working/LTX-Video
!pip install -q -e ".[inference]" 2>&1 | tail -3

## 3. Первый клип: оживляем кадр из зала

Веса `ltxv-2b-0.9.8-distilled` (≈9 ГБ) скачаются автоматически из Hugging Face при первом запуске.

Ключевые параметры:
- `--height 1216 --width 704` — вертикаль 9:16, кратна 32 (требование модели)
- `--num_frames 121` — ~4 секунды при 30 fps
- `--conditioning_media_paths` — кадр-основа, `--conditioning_start_frames 0` — он первый кадр
- `--pipeline_config configs/ltxv-2b-0.9.8-distilled.yaml` — лёгкая и быстрая версия

**Промпт пишите по-английски** и описывайте **движение**, а не сцену: что происходит, куда
движется камера, как ведёт себя свет.

In [ ]:
import os, glob, pathlib

# ==== свои кадры: либо из репозитория, либо загрузите свои через Data -> Upload ====
IMAGES = sorted(glob.glob("/kaggle/working/Open-Generative-AI/mpt-materials/*.png"))
if not IMAGES:
    IMAGES = sorted(glob.glob("/kaggle/input/**/*.png", recursive=True))[:4]
print("кадры:", IMAGES or "НЕ НАЙДЕНЫ — загрузите картинки через панель Data")

PROMPT = ("Slow cinematic push-in as chalk dust drifts through shafts of hard light; "
          "the athlete's shoulders rise with breath, focus sharpening, film grain, "
          "shallow depth of field, moody teal and amber grade")

OUT = pathlib.Path("/kaggle/working/clips"); OUT.mkdir(exist_ok=True)

In [ ]:
%%time
!python inference.py \
  --prompt "{PROMPT}" \
  --conditioning_media_paths "{IMAGES[0]}" \
  --conditioning_start_frames 0 \
  --height 1216 --width 704 --num_frames 121 \
  --seed 42 \
  --pipeline_config configs/ltxv-2b-0.9.8-distilled.yaml \
  --output_path "{OUT}/clip1" \
  2>&1 | tail -15

In [ ]:
import glob
made = sorted(glob.glob(f"{OUT}/clip1/**/*.mp4", recursive=True))
print(made)

## 4. Остальные кадры одним проходом

Каждый клип на T4 — примерно 1.5–4 минуты. Один раз поменяв `PROMPT` на словах движения —
получите серию планов одной сцены.

In [ ]:
%%time
PROMPTS = {
    1: "Slow cinematic push-in as chalk dust drifts through shafts of hard light, athlete's shoulders rising with breath",
    2: "Handheld camera drifts right while sweat droplets fall in slow motion, rim light flickering across the steel bar",
    3: "Locked-off shot, steam rising around the barbell, subtle vibration of the plates as weight settles, dust in the air",
    4: "Slow crane up revealing the dark gym, light beams sweeping across empty racks, dust motes floating",
}

for i, img in enumerate(IMAGES[1:], start=2):
    prompt = PROMPTS.get(i, PROMPTS[1]).replace('"', '')
    print(f"\n=== клип {i}: {os.path.basename(img)} ===")
    !python inference.py \
      --prompt "{prompt}" \
      --conditioning_media_paths "{img}" \
      --conditioning_start_frames 0 \
      --height 1216 --width 704 --num_frames 121 \
      --seed {42 + i} \
      --pipeline_config configs/ltxv-2b-0.9.8-distilled.yaml \
      --output_path "{OUT}/clip{i}" 2>&1 | tail -3

## 5. Склейка и русская озвучка

Собираем клипы в один вертикальный ролик и накладываем голос через Edge TTS
(бесплатно, без ключа: `ru-RU-DmitryNeural`).

In [ ]:
!pip install -q edge-tts imageio-ffmpeg
import glob, subprocess, imageio_ffmpeg

ff = imageio_ffmpeg.get_ffmpeg_exe()
clips = sorted(glob.glob(f"{OUT}/clip*/**/*.mp4", recursive=True))
print("клипов:", len(clips))

with open("/kaggle/working/list.txt", "w") as f:
    for c in clips:
        f.write(f"file '{c}'\n")
subprocess.run([ff, "-y", "-v", "error", "-f", "concat", "-safe", "0",
                "-i", "/kaggle/working/list.txt", "-c", "copy",
                "/kaggle/working/joined.mp4"], check=True)
print("склеено:", os.path.getsize("/kaggle/working/joined.mp4") / 1e6, "МБ")

In [ ]:
SCRIPT = ("Каждый раз, когда ты приходишь в зал, ты выбираешь себя. "
          "Не мотивация держит тебя здесь, а привычка. Один подход. Ещё один. "
          "Начни с малого: тридцать минут, три раза в неделю.")

import asyncio, edge_tts
async def tts():
    await edge_tts.Communicate(SCRIPT, "ru-RU-DmitryNeural").save("/kaggle/working/voice.mp3")
asyncio.run(tts())

In [ ]:
# голос + видео: видео удлиняем/подрезаем под длительность озвучки
import imageio_ffmpeg, subprocess, json
ff = imageio_ffmpeg.get_ffmpeg_exe()
dur = float(subprocess.run([ff, "-v", "error", "-i", "/kaggle/working/voice.mp3",
                            "-f", "null", "-"], capture_output=True, text=True).stderr.count("")) or 0
probe = subprocess.run([ff, "-i", "/kaggle/working/voice.mp3"], capture_output=True, text=True).stderr
import re
m = re.search(r"Duration: (\d+):(\d+):([\d.]+)", probe)
voice_len = int(m.group(1))*3600 + int(m.group(2))*60 + float(m.group(3)) if m else 30
print("длина озвучки:", round(voice_len, 1), "с")

subprocess.run([ff, "-y", "-v", "error",
                "-stream_loop", "-1", "-i", "/kaggle/working/joined.mp4",
                "-i", "/kaggle/working/voice.mp3",
                "-map", "0:v", "-map", "1:a",
                "-t", f"{voice_len:.1f}",
                "-c:v", "libx264", "-preset", "veryfast", "-crf", "20",
                "-c:a", "aac", "-b:a", "160k", "-pix_fmt", "yuv420p",
                "/kaggle/working/reel-9x16.mp4"], check=True)
print("готово: /kaggle/working/reel-9x16.mp4")

## 6. Если хочется качества выше (13B) — но дольше

`ltxv-13b-0.9.8-distilled` даёт заметно лучше детализацию. В 16 ГБ целиком не влезает,
поэтому включаем выгрузку лишнего в RAM/CPU. На T4 это будет медленно (10+ минут на клип),
на P100 — терпимо. Запускайте, только если время не жалко.

```
!python inference.py --prompt "..." \
  --conditioning_media_paths "{IMAGES[0]}" --conditioning_start_frames 0 \
  --height 1216 --width 704 --num_frames 121 --seed 42 --offload_to_cpu \
  --pipeline_config configs/ltxv-13b-0.9.8-distilled.yaml \
  --output_path "{OUT}/clip13b"
```

## 7. Что забрать на выходе

- `/kaggle/working/reel-9x16.mp4` — вертикальный ролик с озвучкой (готов к Reels/Shorts)
- `/kaggle/working/clips/clip*/` — отдельные клипы для монтажа

Скачать: панель **Output** справа → файл → Download.

⚠️ Kaggle ограничивает сессию GPU (до 9–12 часов) и **удаляет рабочую папку между сессиями** —
скачивайте результат сразу или сохраняйте в Kaggle Dataset / GitHub.